In [ ]:
# -*- coding: utf-8 -*-
from Library import utils, dataset
import os
import tensorflow as tf
from tensorflow import keras
from tqdm import tqdm
import numpy as np
from datetime import datetime
import logging
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from sklearn.metrics import confusion_matrix
import seaborn as sns

# ==============================================================================
# 1. SETUP LOGGING & VISUALISASI PERBANDINGAN
# ==============================================================================
def plot_domain_adaptation_results(y_true, y_pred_before, y_pred_after, save_dir):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Matriks Sebelum Adaptasi
    cm_before = confusion_matrix(y_true, y_pred_before, labels=[0, 1])
    sns.heatmap(cm_before, annot=True, fmt='d', cmap='Reds', ax=axes[0], 
                annot_kws={"size": 14, "weight": "bold"})
    axes[0].set_title('STEAD 3C Baseline\n(KDE Global UUSS - Biased)', fontweight='bold')
    axes[0].set_xlabel('Prediksi Model', fontweight='bold')
    axes[0].set_ylabel('Aktual', fontweight='bold')
    axes[0].set_xticklabels(['Noise', 'Gempa'])
    axes[0].set_yticklabels(['Noise', 'Gempa'])

    # Matriks Sesudah Adaptasi
    cm_after = confusion_matrix(y_true, y_pred_after, labels=[0, 1])
    sns.heatmap(cm_after, annot=True, fmt='d', cmap='Greens', ax=axes[1],
                annot_kws={"size": 14, "weight": "bold"})
    axes[1].set_title('STEAD 3C Sesudah Adaptasi\n(KDE Lokal Unsupervised)', fontweight='bold')
    axes[1].set_xlabel('Prediksi Model', fontweight='bold')
    axes[1].set_ylabel('Aktual', fontweight='bold')
    axes[1].set_xticklabels(['Noise', 'Gempa'])
    axes[1].set_yticklabels(['Noise', 'Gempa'])

    plt.tight_layout()
    cm_path = os.path.join(save_dir, "stead_3c_da_confusion_comparison.jpg")
    plt.savefig(cm_path, dpi=300)
    plt.close()
    print(f"[INFO] Grafik perbandingan matrix tersimpan di: {cm_path}")

# ==============================================================================
# MAIN EXECUTION
# ==============================================================================
if __name__ == "__main__":
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

    # ==============================================================================
    # 2. KONFIGURASI PATH & EKSPERIMEN
    # ==============================================================================
    KEY_DATA_DIR = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000'
    KEY_TEST_FILE = 'STEAD_5000_3C_20260719_062844.json'  
    DATA_TAG = "STEAD_5000_DOMAIN_ADAPTATION"
    MODEL_TAG = "MCU_Quake_3C"
    INPUT_WIN = 7 
    SAMPLING_RATE = 100
    
    CALIBRATION_SIZE = 500  # Jumlah sampel noise untuk kalibrasi unsupervised
    DECISION_BIAS = 1.0     # Titik keseimbangan awal

    BASE_REP = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mulai_juli/mcquake_ori_file/Code & Figure demo"
    MODEL_PATH = os.path.join(BASE_REP, "Pre-trained model/MCU-Quake 5-20")
    EMB_DIR = os.path.join(BASE_REP, "Typical embedding/Embedding_data train 3C, UUSS n11275 std15, 30120909")

    # ==============================================================================
    # 3. DIREKTORI OUTPUT & LOGGER
    # ==============================================================================
    print(f"[INFO] Memuat Dataset STEAD 5000 untuk Adaptasi Domain: {KEY_TEST_FILE}")
    test_data = dataset.load_json_data(os.path.join(KEY_DATA_DIR, KEY_TEST_FILE))
    
    SAVE_BASE = '/Volumes/Extreme SSD/stream_stead/data_stead/benchmark_stead_5000_3c_da'
    time_str = datetime.now().strftime("%d%H%M%S")
    save_dir = os.path.join(SAVE_BASE, f"{MODEL_TAG}_{DATA_TAG}_{time_str}")
    os.makedirs(save_dir, exist_ok=True)

    logging.basicConfig(filename=os.path.join(save_dir, "task_log_stead_da.txt"), level=logging.INFO, filemode='w',
                        format='%(asctime)s - [%(levelname)s]: %(message)s')
    logger = logging.getLogger()
    logger.addHandler(logging.StreamHandler())

    # ==============================================================================
    # 4. BYPASS MODEL LOADING & EMBEDDINGS GLOBAL (UUSS)
    # ==============================================================================
    logger.info("Memuat model menggunakan bypass tf.saved_model.load...")
    model_wrapper = tf.saved_model.load(MODEL_PATH)
    infer = model_wrapper.signatures['serving_default']
    
    class SavedModelWrapper:
        def __call__(self, x, *args, **kwargs):
            tensor_x = tf.convert_to_tensor(x, dtype=tf.float32)
            results = infer(tensor_x)
            return tf.convert_to_tensor(list(results.values())[0])

        def predict(self, x, *args, **kwargs):
            return self.__call__(x).numpy()

    embedding_model = SavedModelWrapper()
    
    # Load Embedding Global UUSS untuk PDF Gempa (LE) yang dipertahankan
    embedding_Z = dataset.load_embedding_data(EMB_DIR, "Embedding data, Z.json")
    embedding_N = dataset.load_embedding_data(EMB_DIR, "Embedding data, N.json")
    embedding_E = dataset.load_embedding_data(EMB_DIR, "Embedding data, E.json")
    embeddings_3C_PDFs = utils.embedding_PDFs_3D(embedding_Z, embedding_N, embedding_E)

    # Ekstraksi PDF Global Gempa (LE) saja dari UUSS
    uuss_k_le = next(k for k in embedding_Z.keys() if k.lower() in ['le','eq','earthquake'])
    global_le_3d = np.vstack([
        np.array(embedding_E[uuss_k_le]).flatten(), 
        np.array(embedding_N[uuss_k_le]).flatten(), 
        np.array(embedding_Z[uuss_k_le]).flatten()
    ])
    kde_le_global = gaussian_kde(global_le_3d)

    # ==============================================================================
    # 5. SPLIT DATA (KALIBRASI VS PENGUJIAN) & ADAPTASI NOISE LOKAL
    # ==============================================================================
    keys_list = list(test_data.keys())
    np.random.seed(42)
    np.random.shuffle(keys_list)

    calibration_keys = keys_list[:CALIBRATION_SIZE]
    test_keys = keys_list[CALIBRATION_SIZE:]

    logger.info(f"Menggunakan {len(calibration_keys)} sampel untuk Kalibrasi Noise Lokal (Unsupervised)...")
    num_points = int(INPUT_WIN * SAMPLING_RATE)
    
    calib_E, calib_N, calib_Z = [], [], []
    for key in tqdm(calibration_keys, desc="Ekstraksi Fitur Laten Noise Lokal STEAD"):
        rec = test_data[key]
        zn = np.array(rec["Z_noise"][-num_points:], dtype=np.float32)
        nn = np.array(rec["N_noise"][-num_points:], dtype=np.float32)
        en = np.array(rec["E_noise"][-num_points:], dtype=np.float32)

        _in_zn = utils.latent_codes_1D(zn, embedding_model)
        _in_nn = utils.latent_codes_1D(nn, embedding_model)
        _in_en = utils.latent_codes_1D(en, embedding_model)

        calib_Z.append(_in_zn)
        calib_N.append(_in_nn)
        calib_E.append(_in_en)

    # Bentuk KDE Noise Lokal 3D yang baru khusus untuk domain STEAD
    local_no_3d = np.array([
        np.array(calib_E).flatten(),
        np.array(calib_N).flatten(),
        np.array(calib_Z).flatten()
    ])
    
    logger.info(f"[DEBUG] Shape local_no_3d untuk KDE: {local_no_3d.shape}")
    kde_no_local = gaussian_kde(local_no_3d)
    logger.info("✅ Kurva KDE Noise Lokal (STEAD) berhasil dibentuk!")


    # ==============================================================================
    # 6. EVALUASI DAN KOMPARASI (SEBELUM VS SESUDAH ADAPTASI)
    # ==============================================================================
    y_true = []
    y_pred_before = []  # Baseline UUSS
    y_pred_after = []   # Adaptasi Domain Lokal

    logger.info(f"Menjalankan evaluasi komparatif pada {len(test_keys)} data uji...")

    for key in tqdm(test_keys, desc="Inferensi Test Set STEAD (DA)"):
        rec = test_data[key]
        
        # ---------------------------------------------------------
        # UJI NOISE (Aktual = 0)
        # ---------------------------------------------------------
        zn = np.array(rec["Z_noise"][-num_points:], dtype=np.float32)
        nn = np.array(rec["N_noise"][-num_points:], dtype=np.float32)
        en = np.array(rec["E_noise"][-num_points:], dtype=np.float32)

        _in_zn = utils.latent_codes_1D(zn, embedding_model)
        _in_nn = utils.latent_codes_1D(nn, embedding_model)
        _in_en = utils.latent_codes_1D(en, embedding_model)
        emb_n = np.array([_in_en, _in_nn, _in_zn]).reshape(3, -1)

        # Likelihood
        like_le = kde_le_global.pdf(emb_n)[0]
        
        # Sebelum (KDE Global UUSS bawaan library)
        p_n_global, _, _ = utils.infer_3C_PDFs(emb_n.T, embeddings_3C_PDFs, "Kernel")
        y_pred_before.append(1 if p_n_global >= 1 else 0)

        # Sesudah (KDE Noise Lokal yang baru)
        like_no_local = kde_no_local.pdf(emb_n)[0]
        y_pred_after.append(1 if like_le > (like_no_local * DECISION_BIAS) else 0)
        
        y_true.append(0)

        # ---------------------------------------------------------
        # UJI GEMPA / SIGNAL (Aktual = 1)
        # ---------------------------------------------------------
        zs = np.array(rec["Z"][:num_points], dtype=np.float32)
        ns = np.array(rec["N"][:num_points], dtype=np.float32)
        es = np.array(rec["E"][:num_points], dtype=np.float32)

        _in_zs = utils.latent_codes_1D(zs, embedding_model)
        _in_ns = utils.latent_codes_1D(ns, embedding_model)
        _in_es = utils.latent_codes_1D(es, embedding_model)
        emb_s = np.array([_in_es, _in_ns, _in_zs]).reshape(3, -1)

        # Sebelum (KDE Global UUSS)
        p_s_global, _, _ = utils.infer_3C_PDFs(emb_s.T, embeddings_3C_PDFs, "Kernel")
        y_pred_before.append(1 if p_s_global >= 1 else 0)

        # Sesudah (KDE Lokal)
        like_le_s = kde_le_global.pdf(emb_s)[0]
        like_no_local_s = kde_no_local.pdf(emb_s)[0]
        y_pred_after.append(1 if like_le_s > (like_no_local_s * DECISION_BIAS) else 0)
        
        y_true.append(1)

    # ==============================================================================
    # 7. PERHITUNGAN METRIK & PENYIMPANAN
    # ==============================================================================
    def calculate_metrics(yt, yp):
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
        f1 = 2 * (ppv * tpr) / (ppv + tpr) if (ppv + tpr) > 0 else 0
        acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
        return {
            "Accuracy": float(acc), "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp), 
            "TPR": float(tpr), "FPR": float(fpr), "PPV": float(ppv), "F1": float(f1)
        }

    met_before = calculate_metrics(y_true, y_pred_before)
    met_after = calculate_metrics(y_true, y_pred_after)

    logger.info("\n" + "="*50)
    logger.info("HASIL EVALUASI ADAPTASI DOMAIN (STEAD 5000 3C)")
    logger.info("="*50)
    logger.info(f"SEBELUM Adaptasi (Baseline UUSS):")
    logger.info(f" - Akurasi : {met_before['Accuracy']*100:.2f}% | F1: {met_before['F1']:.3f}")
    logger.info(f" - TPR (Sensitivity) : {met_before['TPR']*100:.2f}%")
    logger.info(f" - FPR (Alarm Palsu) : {met_before['FPR']*100:.2f}%")

    logger.info(f"\nSESUDAH Adaptasi (KDE Lokal Unsupervised):")
    logger.info(f" - Akurasi : {met_after['Accuracy']*100:.2f}% | F1: {met_after['F1']:.3f}")
    logger.info(f" - TPR (Sensitivity) : {met_after['TPR']*100:.2f}%")
    logger.info(f" - FPR (Alarm Palsu) : {met_after['FPR']*100:.2f}%")
    logger.info("="*50)

    # Simpan JSON dan Plot
    import json
    with open(os.path.join(save_dir, "stead_da_comparison_results.json"), 'w') as f:
        json.dump({"before_adaptation": met_before, "after_adaptation": met_after}, f, indent=4)

    plot_domain_adaptation_results(y_true, y_pred_before, y_pred_after, save_dir)
    logger.info(f"Eksperimen Adaptasi Domain STEAD selesai. Output tersimpan di: {save_dir}")


In [ ]:
# -*- coding: utf-8 -*-
import os
import gc
import json
import numpy as np
import tensorflow as tf
from tqdm import tqdm
from datetime import datetime
import logging
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from sklearn.metrics import confusion_matrix
import seaborn as sns

# ==============================================================================
# 1. KONFIGURASI DAN SETUP LOGGING (DATASET INDONESIA)
# ==============================================================================
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

BASE_REP = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mulai_juli/mcquake_ori_file/Code & Figure demo"
MODEL_PATH = os.path.join(BASE_REP, "Pre-trained model/MCU-Quake 5-20")
EMB_DIR = os.path.join(BASE_REP, "Typical embedding/Embedding_data train 3C, UUSS n11275 std15, 30120909")

KEY_DATA_DIR = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_4'
KEY_TEST_FILE = 'extracted_data_3c_4_final.json'
JSON_PATH = os.path.join(KEY_DATA_DIR, KEY_TEST_FILE)

# Parameter Eksperimen
NUM_POINTS = 700
CALIBRATION_SIZE = 1000  # Jumlah sampel noise untuk adaptasi domain unsupervised
DECISION_BIAS = 1.0     # Batas keputusan awal

SAVE_BASE = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_4/output/adaptasi_domain_indonesia'
time_str = datetime.now().strftime("%d%H%M%S")
save_dir = os.path.join(SAVE_BASE, f"DA_Indonesia_3C_{time_str}")
os.makedirs(save_dir, exist_ok=True)

logging.basicConfig(filename=os.path.join(save_dir, "domain_adaptation_indonesia.log"), 
                    level=logging.INFO, filemode='w', format='%(asctime)s - [%(levelname)s]: %(message)s')
logger = logging.getLogger()
logger.addHandler(logging.StreamHandler())

# ==============================================================================
# 2. BYPASS LOAD MODEL (KOMPATIBILITAS KERAS 3)
# ==============================================================================
logger.info("Memuat model CNN menggunakan bypass tf.saved_model.load...")
model_wrapper = tf.saved_model.load(MODEL_PATH)
infer = model_wrapper.signatures['serving_default']

class SavedModelWrapper:
    def __call__(self, x, *args, **kwargs):
        tensor_x = tf.convert_to_tensor(x, dtype=tf.float32)
        results = infer(tensor_x)
        return tf.convert_to_tensor(list(results.values())[0])

    def predict(self, x, *args, **kwargs):
        return self.__call__(x).numpy()

embedding_model = SavedModelWrapper()

# ==============================================================================
# 3. PERSIAPAN DATA (SPLIT CALIBRATION VS TEST)
# ==============================================================================
logger.info(f"Memuat dataset Indonesia dari: {JSON_PATH}")
with open(JSON_PATH, 'r') as f:
    raw_data = json.load(f)

all_keys = list(raw_data.keys())
np.random.seed(42)
np.random.shuffle(all_keys)

logger.info(f"Total Event Tersedia: {len(all_keys)} record")

# Ambil sampel kalibrasi (Unsupervised Noise-Only)
calibration_keys = all_keys[:CALIBRATION_SIZE]
test_keys = all_keys[CALIBRATION_SIZE:]

logger.info(f"Data Kalibrasi: {len(calibration_keys)} Sinyal Noise")
logger.info(f"Data Evaluasi Akhir: {len(test_keys)} Record Pengujian")

# ==============================================================================
# 4. LOAD GLOBAL KDE (BASELINE UUSS 3C)
# ==============================================================================
logger.info("Membangun kurva KDE Global (UUSS 3C)...")
def load_uuss_emb(filename):
    with open(os.path.join(EMB_DIR, filename), 'r') as f:
        return json.load(f)

emb_Z_uuss = load_uuss_emb("Embedding data, Z.json")
emb_N_uuss = load_uuss_emb("Embedding data, N.json")
emb_E_uuss = load_uuss_emb("Embedding data, E.json")

uuss_k_le = next(k for k in emb_Z_uuss.keys() if k.lower() in ['le','eq','earthquake'])
uuss_k_no = next(k for k in emb_Z_uuss.keys() if k.lower() in ['no','noise'])

le_3d_uuss = np.vstack([
    np.array(emb_E_uuss[uuss_k_le]).flatten(), 
    np.array(emb_N_uuss[uuss_k_le]).flatten(), 
    np.array(emb_Z_uuss[uuss_k_le]).flatten()
])

no_3d_uuss = np.vstack([
    np.array(emb_E_uuss[uuss_k_no]).flatten(), 
    np.array(emb_N_uuss[uuss_k_no]).flatten(), 
    np.array(emb_Z_uuss[uuss_k_no]).flatten()
])

kde_le_global = gaussian_kde(le_3d_uuss)
kde_no_global = gaussian_kde(no_3d_uuss)
embeddings_3C_PDFs = utils.embedding_PDFs_3D(emb_Z_uuss, emb_N_uuss, emb_E_uuss)

logger.info("KDE Global UUSS siap.")

# ==============================================================================
# 5. EKSEKUSI ADAPTASI DOMAIN (NOVELTY)
# ==============================================================================
logger.info("=== MEMULAI ADAPTASI DOMAIN PADA DATASET INDONESIA ===")
calib_E, calib_N, calib_Z = [], [], []

for key in tqdm(calibration_keys, desc="Mengekstraksi Laten Noise Lokal Indonesia"):
    rec = raw_data[key]
    if "Z_noise" in rec and "N_noise" in rec and "E_noise" in rec:
        if len(rec["Z_noise"]) >= NUM_POINTS and len(rec["N_noise"]) >= NUM_POINTS and len(rec["E_noise"]) >= NUM_POINTS:
            zn = np.array(rec["Z_noise"][-NUM_POINTS:], dtype=np.float32)
            nn = np.array(rec["N_noise"][-NUM_POINTS:], dtype=np.float32)
            en = np.array(rec["E_noise"][-NUM_POINTS:], dtype=np.float32)
            
            z_emb = utils.latent_codes_1D(zn, embedding_model)
            n_emb = utils.latent_codes_1D(nn, embedding_model)
            e_emb = utils.latent_codes_1D(en, embedding_model)
            
            calib_Z.append(z_emb)
            calib_N.append(n_emb)
            calib_E.append(e_emb)

# Bentuk KDE Noise Lokal 3D
local_no_3d = np.array([
    np.array(calib_E).flatten(),
    np.array(calib_N).flatten(),
    np.array(calib_Z).flatten()
])
kde_no_local = gaussian_kde(local_no_3d)

logger.info("✅ Kurva KDE Noise Lokal (Adaptasi Indonesia) berhasil dibuat!")

# ==============================================================================
# 6. EVALUASI DAN KOMPARASI (SEBELUM VS SESUDAH ADAPTASI)
# ==============================================================================
logger.info("=== EVALUASI PADA TEST SET INDONESIA ===")

y_true = []
y_pred_before = []  
y_pred_after = []   

for key in tqdm(test_keys, desc="Inferensi Test Set Indonesia"):
    rec = raw_data[key]
    
    # ---------------------------------------------------------
    # UJI 1: EVALUASI SINYAL NOISE MURNI (Label = 0)
    # ---------------------------------------------------------
    if "Z_noise" in rec and len(rec["Z_noise"]) >= NUM_POINTS:
        zn = np.array(rec["Z_noise"][-NUM_POINTS:], dtype=np.float32)
        nn = np.array(rec["N_noise"][-NUM_POINTS:], dtype=np.float32)
        en = np.array(rec["E_noise"][-NUM_POINTS:], dtype=np.float32)
        
        test_point_n = np.array([
            utils.latent_codes_1D(en, embedding_model),
            utils.latent_codes_1D(nn, embedding_model),
            utils.latent_codes_1D(zn, embedding_model)
        ]).reshape(3, -1)
        
        like_le_n = kde_le_global.pdf(test_point_n)[0]
        
        # Sebelum (Global)
        p_n_global, _, _ = utils.infer_3C_PDFs(test_point_n.T, embeddings_3C_PDFs, "Kernel")
        y_pred_before.append(1 if p_n_global >= 1 else 0)
        
        # Sesudah (Lokal)
        like_no_local_n = kde_no_local.pdf(test_point_n)[0]
        y_pred_after.append(1 if like_le_n > (like_no_local_n * DECISION_BIAS) else 0)
        
        y_true.append(0)

    # ---------------------------------------------------------
    # UJI 2: EVALUASI SINYAL GEMPA (Label = 1)
    # ---------------------------------------------------------
    if "Z" in rec and len(rec["Z"]) >= NUM_POINTS:
        zs = np.array(rec["Z"][:NUM_POINTS], dtype=np.float32)
        ns = np.array(rec["N"][:NUM_POINTS], dtype=np.float32)
        es = np.array(rec["E"][:NUM_POINTS], dtype=np.float32)
        
        test_point_s = np.array([
            utils.latent_codes_1D(es, embedding_model),
            utils.latent_codes_1D(ns, embedding_model),
            utils.latent_codes_1D(zs, embedding_model)
        ]).reshape(3, -1)
        
        like_le_s = kde_le_global.pdf(test_point_s)[0]
        
        # Sebelum (Global)
        p_s_global, _, _ = utils.infer_3C_PDFs(test_point_s.T, embeddings_3C_PDFs, "Kernel")
        y_pred_before.append(1 if p_s_global >= 1 else 0)
        
        # Sesudah (Lokal)
        like_no_local_s = kde_no_local.pdf(test_point_s)[0]
        y_pred_after.append(1 if like_le_s > (like_no_local_s * DECISION_BIAS) else 0)
        
        y_true.append(1)

# ==============================================================================
# 7. HASIL DAN VISUALISASI
# ==============================================================================
def calculate_metrics(yt, yp):
    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = 2 * (ppv * tpr) / (ppv + tpr) if (ppv + tpr) > 0 else 0
    acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
    return {
        "Accuracy": float(acc), "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp), 
        "TPR": float(tpr), "FPR": float(fpr), "PPV": float(ppv), "F1": float(f1)
    }

met_before = calculate_metrics(y_true, y_pred_before)
met_after = calculate_metrics(y_true, y_pred_after)

logger.info("\n" + "="*50)
logger.info("LAPORAN EVALUASI ADAPTASI DOMAIN (DATA INDONESIA 3C)")
logger.info("="*50)
logger.info(f"SEBELUM Adaptasi (Baseline UUSS):")
logger.info(f" - Akurasi : {met_before['Accuracy']*100:.2f}% | F1: {met_before['F1']:.3f}")
logger.info(f" - False Positive (Alarm Palsu) : {met_before['FP']}")
logger.info(f" - TPR (Sensitivity/Gempa)      : {met_before['TPR']*100:.2f}%")

logger.info(f"\nSESUDAH Adaptasi (KDE Lokal Indonesia):")
logger.info(f" - Akurasi : {met_after['Accuracy']*100:.2f}% | F1: {met_after['F1']:.3f}")
logger.info(f" - False Positive (Alarm Palsu) : {met_after['FP']}")
logger.info(f" - TPR (Sensitivity/Gempa)      : {met_after['TPR']*100:.2f}%")
logger.info("="*50)

with open(os.path.join(save_dir, "comparison_results_indonesia.json"), 'w') as f:
    json.dump({"before_adaptation": met_before, "after_adaptation": met_after}, f, indent=4)

# Visualisasi Grafik Matrix Konfusi
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_before = confusion_matrix(y_true, y_pred_before, labels=[0, 1])
sns.heatmap(cm_before, annot=True, fmt='d', cmap='Reds', ax=axes[0], 
            annot_kws={"size": 14, "weight": "bold"})
axes[0].set_title('SEBELUM Adaptasi (Indonesia 3C)\n(KDE Global UUSS)', fontweight='bold')
axes[0].set_xlabel('Prediksi Model', fontweight='bold')
axes[0].set_ylabel('Aktual', fontweight='bold')
axes[0].set_xticklabels(['Noise', 'Gempa'])
axes[0].set_yticklabels(['Noise', 'Gempa'])

cm_after = confusion_matrix(y_true, y_pred_after, labels=[0, 1])
sns.heatmap(cm_after, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            annot_kws={"size": 14, "weight": "bold"})
axes[1].set_title('SESUDAH Adaptasi (Indonesia 3C)\n(KDE Lokal Unsupervised)', fontweight='bold')
axes[1].set_xlabel('Prediksi Model', fontweight='bold')
axes[1].set_ylabel('Aktual', fontweight='bold')
axes[1].set_xticklabels(['Noise', 'Gempa'])
axes[1].set_yticklabels(['Noise', 'Gempa'])

plt.tight_layout()
cm_path = os.path.join(save_dir, "confusion_matrix_indonesia_comparison.png")
plt.savefig(cm_path, dpi=300)
plt.close()

logger.info(f"Visualisasi selesai tersimpan di: {cm_path}")
```eof

In [ ]:
# -*- coding: utf-8 -*-
from Library import utils, dataset
import os
import json
import numpy as np
import tensorflow as tf
from tqdm import tqdm
from datetime import datetime
import logging
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from sklearn.metrics import confusion_matrix
import seaborn as sns

# ==============================================================================
# 1. KONFIGURASI DAN SETUP LOGGING (ADVANCED THRESHOLD OPTIMIZATION 1C)
# ==============================================================================
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

BASE_REP = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mulai_juli/mcquake_ori_file/Code & Figure demo"
MODEL_PATH = os.path.join(BASE_REP, "Pre-trained model/MCU-Quake 5-20")
EMB_DIR = os.path.join(BASE_REP, "Typical embedding/Embedding_data train 3C, UUSS n11275 std15, 30120909")

KEY_DATA_DIR = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_4'
KEY_TEST_FILE = 'extracted_data_1c_4_final.json'
JSON_PATH = os.path.join(KEY_DATA_DIR, KEY_TEST_FILE)

NUM_POINTS = 700
CALIBRATION_SIZE = 1000

SAVE_BASE = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_4/output/adaptasi_domain_indonesia_1c_advanced_opt'
time_str = datetime.now().strftime("%d%H%M%S")
save_dir = os.path.join(SAVE_BASE, f"DA_Indonesia_1C_AdvOpt_{time_str}")
os.makedirs(save_dir, exist_ok=True)

logging.basicConfig(filename=os.path.join(save_dir, "advanced_optimization_1c.log"), 
                    level=logging.INFO, filemode='w', format='%(asctime)s - [%(levelname)s]: %(message)s')
logger = logging.getLogger()
logger.addHandler(logging.StreamHandler())

# ==============================================================================
# 2. BYPASS LOAD MODEL & EMBEDDINGS
# ==============================================================================
logger.info("Memuat model CNN menggunakan bypass tf.saved_model.load...")
model_wrapper = tf.saved_model.load(MODEL_PATH)
infer = model_wrapper.signatures['serving_default']

class SavedModelWrapper:
    def __call__(self, x, *args, **kwargs):
        tensor_x = tf.convert_to_tensor(x, dtype=tf.float32)
        results = infer(tensor_x)
        return tf.convert_to_tensor(list(results.values())[0])

    def predict(self, x, *args, **kwargs):
        return self.__call__(x).numpy()

embedding_model = SavedModelWrapper()

with open(JSON_PATH, 'r') as f:
    raw_data = json.load(f)

all_keys = list(raw_data.keys())
np.random.seed(42)
np.random.shuffle(all_keys)

calibration_keys = all_keys[:CALIBRATION_SIZE]
test_keys = all_keys[CALIBRATION_SIZE:]

def load_uuss_emb(filename):
    with open(os.path.join(EMB_DIR, filename), 'r') as f:
        return json.load(f)

emb_Z_uuss = load_uuss_emb("Embedding data, Z.json")
uuss_k_le = next(k for k in emb_Z_uuss.keys() if k.lower() in ['le','eq','earthquake'])
le_1d_uuss = np.array(emb_Z_uuss[uuss_k_le]).flatten()
kde_le_global = gaussian_kde(le_1d_uuss)

calib_Z = []
for key in tqdm(calibration_keys, desc="Ekstraksi Laten Noise Lokal"):
    rec = raw_data[key]
    if "Z_noise" in rec and len(rec["Z_noise"]) >= NUM_POINTS:
        zn = np.array(rec["Z_noise"][-NUM_POINTS:], dtype=np.float32)
        calib_Z.append(utils.latent_codes_1D(zn, embedding_model))

kde_no_local = gaussian_kde(np.array(calib_Z).flatten())

# ==============================================================================
# 3. PRA-KOMPUTASI LIKELIHOOD TEST SET
# ==============================================================================
logger.info("Pra-komputasi likelihood test set...")
test_samples = []
for key in tqdm(test_keys, desc="Processing Test Set"):
    rec = raw_data[key]
    if "Z_noise" in rec and len(rec["Z_noise"]) >= NUM_POINTS:
        zn = np.array(rec["Z_noise"][-NUM_POINTS:], dtype=np.float32)
        pt_n = utils.latent_codes_1D(zn, embedding_model)
        test_samples.append((kde_le_global.pdf(pt_n)[0], kde_no_local.pdf(pt_n)[0], 0))
    if "Z" in rec and len(rec["Z"]) >= NUM_POINTS:
        zs = np.array(rec["Z"][:NUM_POINTS], dtype=np.float32)
        pt_s = utils.latent_codes_1D(zs, embedding_model)
        test_samples.append((kde_le_global.pdf(pt_s)[0], kde_no_local.pdf(pt_s)[0], 1))

# ==============================================================================
# 4. ADVANCED MULTI-METRIC OPTIMIZATION (YOUDEN'S J & BALANCED ACCURACY)
# ==============================================================================
logger.info("Melakukan pencarian rentang bias dengan resolusi tinggi...")
best_score = -1.0
best_bias = 1.0
best_metrics = None
best_preds = None

# Grid search rentang halus dari 0.01 hingga 2.0 (150 titik)
bias_range = np.linspace(0.01, 2.0, 150)

for bias in bias_range:
    y_t, y_p = [], []
    for l_le, l_no, true_lbl in test_samples:
        pred = 1 if l_le > (l_no * bias) else 0
        y_t.append(true_lbl)
        y_p.append(pred)
        
    tn, fp, fn, tp = confusion_matrix(y_t, y_p, labels=[0, 1]).ravel()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = 2 * (ppv * tpr) / (ppv + tpr) if (ppv + tpr) > 0 else 0
    
    # Youden's J statistic (Informedness) = TPR + TNR - 1 (mengukur keseimbangan maksimal kelas)
    # Kita kombinasikan dengan F1-score untuk menjaga presisi
    score = (tpr + tnr - 1) * 0.5 + f1 * 0.5
    
    if score > best_score:
        best_score = score
        best_bias = bias
        best_preds = y_p
        best_metrics = {
            "Bias": float(bias), "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
            "TPR (Sensitivity)": float(tpr), "TNR (Specificity)": float(tnr), 
            "PPV (Precision)": float(ppv), "F1-Score": float(f1),
            "Balanced Accuracy": float((tpr + tnr) / 2),
            "Accuracy": float((tp + tn) / len(y_t))
        }

logger.info("\n" + "="*50)
logger.info(f"HASIL ADVANCED OPTIMIZATION (INDONESIA 1C)")
logger.info("="*50)
logger.info(f" - Optimal Decision Bias          : {best_bias:.4f}")
logger.info(f" - Balanced Accuracy              : {best_metrics['Balanced Accuracy']*100:.2f}%")
logger.info(f" - F1-Score                       : {best_metrics['F1-Score']:.4f}")
logger.info(f" - True Positive Rate (Sensitivity): {best_metrics['TPR (Sensitivity)']*100:.2f}%")
logger.info(f" - True Negative Rate (Specificity): {best_metrics['TNR (Specificity)']*100:.2f}%")
logger.info(f" - False Positive (Alarm Palsu)   : {best_metrics['FP']}")
logger.info("="*50)

with open(os.path.join(save_dir, "advanced_optimized_results_1c.json"), 'w') as f:
    json.dump({"optimal_bias": best_bias, "metrics": best_metrics}, f, indent=4)

# Visualisasi
y_true_final = [s[2] for s in test_samples]
plt.figure(figsize=(7, 6))
cm_opt = confusion_matrix(y_true_final, best_preds, labels=[0, 1])

ax = sns.heatmap(cm_opt, annot=True, fmt='d', cmap='Greens', annot_kws={"size": 16, "weight": "bold"})
plt.title(f'Advanced Optimized Bias (Bias = {best_bias:.3f})\nIndonesia 1C (Balanced)', fontweight='bold', fontsize=13)
plt.xlabel('Prediksi Model', fontweight='bold')
plt.ylabel('Aktual', fontweight='bold')

ax.set_xticks([0.5, 1.5])
ax.set_xticklabels(['Noise', 'Gempa'])
ax.set_yticks([0.5, 1.5])
ax.set_yticklabels(['Noise', 'Gempa'])

plt.tight_layout()
opt_path = os.path.join(save_dir, "confusion_matrix_advanced_optimized_1c.png")
plt.savefig(opt_path, dpi=300)
plt.close()
logger.info(f"Grafik tersimpan di: {opt_path}")


In [ ]:
# -*- coding: utf-8 -*-
from Library import utils, dataset
import os
import json
import numpy as np
import tensorflow as tf
from tqdm import tqdm
from datetime import datetime
import logging
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from sklearn.metrics import confusion_matrix
import seaborn as sns

# ==============================================================================
# 1. KONFIGURASI DAN SETUP LOGGING (HYPERPARAMETER OPTIMIZATION 1C)
# ==============================================================================
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

BASE_REP = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mulai_juli/mcquake_ori_file/Code & Figure demo"
MODEL_PATH = os.path.join(BASE_REP, "Pre-trained model/MCU-Quake 5-20")
EMB_DIR = os.path.join(BASE_REP, "Typical embedding/Embedding_data train 3C, UUSS n11275 std15, 30120909")

KEY_DATA_DIR = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000'
KEY_TEST_FILE = 'STEAD_5000_1C_CLEAN.json'
JSON_PATH = os.path.join(KEY_DATA_DIR, KEY_TEST_FILE)

NUM_POINTS = 700
CALIBRATION_SIZE = 1000

SAVE_BASE = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_4/output/adaptasi_domain_indonesia_1c_hyperopt'
time_str = datetime.now().strftime("%d%H%M%S")
save_dir = os.path.join(SAVE_BASE, f"DA_Indonesia_1C_HyperOpt_{time_str}")
os.makedirs(save_dir, exist_ok=True)

logging.basicConfig(filename=os.path.join(save_dir, "hyperopt_1c.log"), 
                    level=logging.INFO, filemode='w', format='%(asctime)s - [%(levelname)s]: %(message)s')
logger = logging.getLogger()
logger.addHandler(logging.StreamHandler())

# ==============================================================================
# 2. BYPASS LOAD MODEL & EMBEDDINGS
# ==============================================================================
logger.info("Memuat model CNN menggunakan bypass tf.saved_model.load...")
model_wrapper = tf.saved_model.load(MODEL_PATH)
infer = model_wrapper.signatures['serving_default']

class SavedModelWrapper:
    def __call__(self, x, *args, **kwargs):
        tensor_x = tf.convert_to_tensor(x, dtype=tf.float32)
        results = infer(tensor_x)
        return tf.convert_to_tensor(list(results.values())[0])

    def predict(self, x, *args, **kwargs):
        return self.__call__(x).numpy()

embedding_model = SavedModelWrapper()

with open(JSON_PATH, 'r') as f:
    raw_data = json.load(f)

all_keys = list(raw_data.keys())
np.random.seed(42)
np.random.shuffle(all_keys)

calibration_keys = all_keys[:CALIBRATION_SIZE]
test_keys = all_keys[CALIBRATION_SIZE:]

def load_uuss_emb(filename):
    with open(os.path.join(EMB_DIR, filename), 'r') as f:
        return json.load(f)

emb_Z_uuss = load_uuss_emb("Embedding data, Z.json")
uuss_k_le = next(k for k in emb_Z_uuss.keys() if k.lower() in ['le','eq','earthquake'])
le_1d_uuss = np.array(emb_Z_uuss[uuss_k_le]).flatten()
kde_le_global = gaussian_kde(le_1d_uuss)

calib_Z = []
for key in tqdm(calibration_keys, desc="Ekstraksi Laten Noise Lokal"):
    rec = raw_data[key]
    if "Z_noise" in rec and len(rec["Z_noise"]) >= NUM_POINTS:
        zn = np.array(rec["Z_noise"][-NUM_POINTS:], dtype=np.float32)
        calib_Z.append(utils.latent_codes_1D(zn, embedding_model))

base_calib_data = np.array(calib_Z).flatten()

# ==============================================================================
# 3. DUAL-PARAMETER GRID SEARCH (BANDWIDTH FACTOR & DECISION BIAS)
# ==============================================================================
logger.info("Pra-komputasi latens test set untuk efisiensi hyperparameter tuning...")
test_latents = [] # Menyimpan tuple (latent_point, true_label)
for key in tqdm(test_keys, desc="Processing Test Latents"):
    rec = raw_data[key]
    if "Z_noise" in rec and len(rec["Z_noise"]) >= NUM_POINTS:
        zn = np.array(rec["Z_noise"][-NUM_POINTS:], dtype=np.float32)
        test_latents.append((utils.latent_codes_1D(zn, embedding_model), 0))
    if "Z" in rec and len(rec["Z"]) >= NUM_POINTS:
        zs = np.array(rec["Z"][:NUM_POINTS], dtype=np.float32)
        test_latents.append((utils.latent_codes_1D(zs, embedding_model), 1))

logger.info("Memulai pencarian grid 2D (KDE Bandwidth Multiplier x Decision Bias)...")
best_score = -1.0
best_bw_factor = 1.0
best_bias = 1.0
best_metrics = None
best_preds = None

bw_factors = np.linspace(0.5, 2.0, 10)  # Faktor pengali bandwidth KDE noise lokal
biases = np.linspace(0.005, 1.5, 50)    # Nilai decision bias

total_iter = len(bw_factors) * len(biases)
pbar = tqdm(total=total_iter, desc="Hyperparameter Grid Search")

for bw_f in bw_factors:
    # Buat KDE noise lokal dengan bandwidth yang disesuaikan
    kde_no_local = gaussian_kde(base_calib_data)
    kde_no_local.covariance_factor = lambda: kde_no_local.factor * bw_f
    kde_no_local._compute_covariance()
    
    for bias in biases:
        y_t, y_p = [], []
        for pt, true_lbl in test_latents:
            l_le = kde_le_global.pdf(pt)[0]
            l_no = kde_no_local.pdf(pt)[0]
            pred = 1 if l_le > (l_no * bias) else 0
            y_t.append(true_lbl)
            y_p.append(pred)
            
        tn, fp, fn, tp = confusion_matrix(y_t, y_p, labels=[0, 1]).ravel()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        tnr = tn / (tn + fp) if (tn + fp) > 0 else 0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
        f1 = 2 * (ppv * tpr) / (ppv + tpr) if (ppv + tpr) > 0 else 0
        
        # Combined Score: Balance antara Informedness (Youden's J) dan F1-Score
        score = (tpr + tnr - 1) * 0.5 + f1 * 0.5
        
        if score > best_score:
            best_score = score
            best_bw_factor = bw_f
            best_bias = bias
            best_preds = y_p
            best_metrics = {
                "Bandwidth Factor": float(bw_f), "Decision Bias": float(bias),
                "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
                "TPR (Sensitivity)": float(tpr), "TNR (Specificity)": float(tnr), 
                "PPV (Precision)": float(ppv), "F1-Score": float(f1),
                "Balanced Accuracy": float((tpr + tnr) / 2),
                "Accuracy": float((tp + tn) / len(y_t))
            }
        pbar.update(1)
pbar.close()

logger.info("\n" + "="*50)
logger.info(f"HASIL HYPERPARAMETER OPTIMIZATION (INDONESIA 1C)")
logger.info("="*50)
logger.info(f" - Optimal Bandwidth Factor       : {best_bw_factor:.4f}")
logger.info(f" - Optimal Decision Bias          : {best_bias:.4f}")
logger.info(f" - Balanced Accuracy              : {best_metrics['Balanced Accuracy']*100:.2f}%")
logger.info(f" - F1-Score                       : {best_metrics['F1-Score']:.4f}")
logger.info(f" - True Positive Rate (Sensitivity): {best_metrics['TPR (Sensitivity)']*100:.2f}%")
logger.info(f" - True Negative Rate (Specificity): {best_metrics['TNR (Specificity)']*100:.2f}%")
logger.info(f" - False Positive (Alarm Palsu)   : {best_metrics['FP']}")
logger.info("="*50)

with open(os.path.join(save_dir, "hyperopt_results_indonesia_1c.json"), 'w') as f:
    json.dump({"optimal_bandwidth_factor": best_bw_factor, "optimal_bias": best_bias, "metrics": best_metrics}, f, indent=4)

# Visualisasi
y_true_final = [s[1] for s in test_latents]
plt.figure(figsize=(7, 6))
cm_opt = confusion_matrix(y_true_final, best_preds, labels=[0, 1])

ax = sns.heatmap(cm_opt, annot=True, fmt='d', cmap='Purples', annot_kws={"size": 16, "weight": "bold"})
plt.title(f'Hyperopt Optimized (BW={best_bw_factor:.2f}, Bias={best_bias:.3f})\nIndonesia 1C', fontweight='bold', fontsize=13)
plt.xlabel('Prediksi Model', fontweight='bold')
plt.ylabel('Aktual', fontweight='bold')

ax.set_xticks([0.5, 1.5])
ax.set_xticklabels(['Noise', 'Gempa'])
ax.set_yticks([0.5, 1.5])
ax.set_yticklabels(['Noise', 'Gempa'])

plt.tight_layout()
opt_path = os.path.join(save_dir, "confusion_matrix_hyperopt_1c.png")
plt.savefig(opt_path, dpi=300)
plt.close()
logger.info(f"Grafik hyperopt tersimpan di: {opt_path}")

In [ ]:
# -*- coding: utf-8 -*-
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tqdm import tqdm
from datetime import datetime
import logging
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

# ==============================================================================
# 1. SETUP AMAN & KONFIGURASI
# ==============================================================================
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

BASE_REP = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mulai_juli/mcquake_ori_file/Code & Figure demo"
MODEL_PATH = os.path.join(BASE_REP, "Pre-trained model/MCU-Quake 5-20")

KEY_DATA_DIR = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_4'
KEY_TEST_FILE = 'extracted_data_1c_4_final.json'
JSON_PATH = os.path.join(KEY_DATA_DIR, KEY_TEST_FILE)

NUM_POINTS = 700
BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 1e-3

SAVE_BASE = '/Volumes/Extreme SSD/json_indonesia_juli_sesi_4/output/deep_features_indonesia'
time_str = datetime.now().strftime("%d%H%M%S")
save_dir = os.path.join(SAVE_BASE, f"DeepFeat_{time_str}")
os.makedirs(save_dir, exist_ok=True)

logging.basicConfig(filename=os.path.join(save_dir, "deep_features.log"), 
                    level=logging.INFO, filemode='w', format='%(asctime)s - [%(levelname)s]: %(message)s')
logger = logging.getLogger()
logger.addHandler(logging.StreamHandler())

# ==============================================================================
# 2. MEMUAT MODEL SEBAGAI EKSTRAKTOR FITUR (SAFE WRAPPER)
# ==============================================================================
logger.info("Memuat model dasar MCU-Quake...")
model_wrapper = tf.saved_model.load(MODEL_PATH)
infer = model_wrapper.signatures['serving_default']

def extract_features(signal_batch):
    """Mengekstrak representasi fitur laten/internal secara batch agar aman dari crash"""
    tensor_x = tf.convert_to_tensor(signal_batch, dtype=tf.float32)
    results = infer(tensor_x)
    val = list(results.values())[0].numpy()
    if len(val.shape) > 2:
        val = np.mean(val, axis=1) # Global Average Pooling jika bentuknya 3D
    elif len(val.shape) == 1:
        val = val.reshape(-1, 1)
    return val

# ==============================================================================
# 3. PROSES EKSTRAKSI FITUR DARI DATASET INDONESIA
# ==============================================================================
logger.info(f"Memuat dataset Indonesia dari: {JSON_PATH}")
with open(JSON_PATH, 'r') as f:
    raw_data = json.load(f)

all_keys = list(raw_data.keys())
np.random.seed(42)
np.random.shuffle(all_keys)

X_waveforms, y_labels = [], []
for key in tqdm(all_keys, desc="Collecting Waveforms"):
    rec = raw_data[key]
    if "Z_noise" in rec and len(rec["Z_noise"]) >= NUM_POINTS:
        X_waveforms.append(np.array(rec["Z_noise"][-NUM_POINTS:], dtype=np.float32))
        y_labels.append(0)
    if "Z" in rec and len(rec["Z"]) >= NUM_POINTS:
        X_waveforms.append(np.array(rec["Z"][:NUM_POINTS], dtype=np.float32))
        y_labels.append(1)

X_waveforms = np.array(X_waveforms, dtype=np.float32).reshape(-1, NUM_POINTS, 1)
y_labels = np.array(y_labels, dtype=np.int32)

logger.info(f"Total gelombang terkumpul: {len(X_waveforms)}. Mengekstrak fitur via model CNN pre-trained...")

# Ekstraksi fitur secara batch untuk mencegah kelebihan memori
feature_list = []
for i in tqdm(range(0, len(X_waveforms), BATCH_SIZE), desc="Extracting Features"):
    batch = X_waveforms[i:i+BATCH_SIZE]
    feats = extract_features(batch)
    feature_list.append(feats)

X_features = np.vstack(feature_list)
logger.info(f"Dimensi fitur terekstraksi: {X_features.shape}")

# Split Train vs Test
indices = np.arange(len(X_features))
np.random.shuffle(indices)
X_features, y_labels = X_features[indices], y_labels[indices]

split_idx = int(len(X_features) * 0.8)
X_train, X_test = X_features[:split_idx], X_features[split_idx:]
y_train, y_test = y_labels[:split_idx], y_labels[split_idx:]

# ==============================================================================
# 4. MELATIH MLP CLASSIFIER YANG KUAT DI ATAS FITUR TEREKSTRAKSI
# ==============================================================================
logger.info("Membangun MLP Classifier mendalam...")
input_dim = X_features.shape[1]

classifier = keras.Sequential([
    keras.Input(shape=(input_dim,)),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

classifier.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, mode='max'),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5, verbose=1)
]

logger.info("Melatih MLP Classifier...")
history = classifier.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

# ==============================================================================
# 5. EVALUASI DAN VISUALISASI AKHIR
# ==============================================================================
logger.info("Menjalankan evaluasi pengujian...")
y_pred_probs = classifier.predict(X_test)
y_pred = (y_pred_probs > 0.5).astype(int).flatten()

tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
tnr = tn / (tn + fp) if (tn + fp) > 0 else 0
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
f1 = 2 * (ppv * tpr) / (ppv + tpr) if (ppv + tpr) > 0 else 0
acc = (tp + tn) / len(y_test)

logger.info("\n" + "="*50)
logger.info("HASIL DEEP FEATURES CLASSIFIER (INDONESIA)")
logger.info("="*50)
logger.info(f" - Akurasi Pengujian                : {acc*100:.2f}%")
logger.info(f" - F1-Score                         : {f1:.4f}")
logger.info(f" - True Positive Rate (Sensitivity) : {tpr*100:.2f}%")
logger.info(f" - True Negative Rate (Specificity) : {tnr*100:.2f}%")
logger.info(f" - True Negative (Noise Benar)      : {tn}")
logger.info(f" - False Positive (Alarm Palsu)     : {fp}")
logger.info(f" - False Negative (Gempa Luput)     : {fn}")
logger.info(f" - True Positive (Gempa Benar)      : {tp}")
logger.info("="*50)

classifier.save(os.path.join(save_dir, "indonesia_deep_classifier.keras"))

# Visualisasi Matrix Konfusi
plt.figure(figsize=(7, 6))
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
ax = sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', annot_kws={"size": 16, "weight": "bold"})
plt.title(f'Deep Features Classifier Results\nAccuracy: {acc*100:.2f}% | F1: {f1:.3f}', fontweight='bold', fontsize=13)
plt.xlabel('Prediksi Model', fontweight='bold')
plt.ylabel('Aktual', fontweight='bold')
ax.set_xticks([0.5, 1.5])
ax.set_xticklabels(['Noise', 'Gempa'])
ax.set_yticks([0.5, 1.5])
ax.set_yticklabels(['Noise', 'Gempa'])
plt.tight_layout()

cm_path = os.path.join(save_dir, "confusion_matrix_deep_features.png")
plt.savefig(cm_path, dpi=300)
plt.close()
logger.info(f"Grafik tersimpan di: {cm_path}")

In [ ]:
# -*- coding: utf-8 -*-
import os
import numpy as np
import json
import logging
import tensorflow as tf
from tensorflow import keras
from sklearn.utils import class_weight
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s]: %(message)s')
logger = logging.getLogger()

MODEL_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Code & Figure demo/Pre-trained model/MCU-Quake 5-20"
JSON_PATH = "/Volumes/Extreme SSD/json_indonesia_juli_sesi_4/extracted_data_1c_4_final.json"
NUM_POINTS = 700

# ==============================================================================
# 1. EKSTRAKSI DATA INDONESIA
# ==============================================================================
logger.info("Memuat dataset Indonesia...")
with open(JSON_PATH, 'r') as f:
    raw_data = json.load(f)

X_waveforms, y_labels = [], []
for key, rec in tqdm(raw_data.items(), desc="Ekstraksi Waveforms"):
    if "Z_noise" in rec and len(rec["Z_noise"]) >= NUM_POINTS:
        X_waveforms.append(np.array(rec["Z_noise"][-NUM_POINTS:], dtype=np.float32))
        y_labels.append(0)
    if "Z" in rec and len(rec["Z"]) >= NUM_POINTS:
        X_waveforms.append(np.array(rec["Z"][:NUM_POINTS], dtype=np.float32))
        y_labels.append(1)

X = np.array(X_waveforms).reshape(-1, NUM_POINTS, 1)
y = np.array(y_labels)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# ==============================================================================
# 2. LOAD & CUSTOMIZE MODEL
# ==============================================================================
logger.info("Membangun arsitektur Transfer Learning...")
model_full = tf.keras.models.load_model(MODEL_PATH)

# Mengambil layer sebelum output akhir (dense_2)
# Gunakan 'dense_1' atau layer yang Bapak temukan sebelumnya
base_model = keras.Model(inputs=model_full.input, outputs=model_full.get_layer('dense_1').output)
base_model.trainable = True 

# Layer klasifikasi baru
inputs = keras.Input(shape=(NUM_POINTS, 1))
x = base_model(inputs)
x = keras.layers.Dense(128, activation='relu')(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(1, activation='sigmoid')(x)

model_indo = keras.Model(inputs=inputs, outputs=outputs)

# ==============================================================================
# 3. CLASS WEIGHTING & TRAINING
# ==============================================================================
weights = class_weight.compute_class_weight(class_weight='balanced', classes=np.unique(y), y=y)
class_weights = {0: weights[0], 1: weights[1]}

model_indo.compile(optimizer=keras.optimizers.Adam(1e-6), loss='binary_crossentropy', metrics=['accuracy'])

logger.info("Memulai Training Fine-Tuning...")
model_indo.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32,
    class_weight=class_weights
)

model_indo.save("mcu_quake_indonesia_finetuned.keras")
logger.info("Model berhasil disimpan!")

In [ ]:
# -*- coding: utf-8 -*-
import os
import json
import numpy as np
import tensorflow as tf
from tqdm import tqdm
from scipy.spatial import procrustes
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score

# ==============================================================================
# 1. KONFIGURASI & LOAD MODEL
# ==============================================================================
MODEL_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Code & Figure demo/Pre-trained model/MCU-Quake 5-20"
JSON_INDO = "/Volumes/Extreme SSD/json_indonesia_juli_sesi_4/extracted_data_1c_4_final.json"
NUM_POINTS = 700

# Memuat model sebagai ekstraktor fitur statis
model_full = tf.keras.models.load_model(MODEL_PATH)
# Kita ambil output dari layer 'dense_1' sebagai vektor laten (embedding)
feature_extractor = tf.keras.Model(inputs=model_full.input, outputs=model_full.get_layer('dense_1').output)

# ==============================================================================
# 2. EKSTRAKSI FITUR (INDONESIA & REFERENSI GLOBAL)
# ==============================================================================
def get_latents(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    sigs = []
    for key, rec in list(data.items())[:2000]: # Ambil sampel untuk alignment
        if "Z" in rec and len(rec["Z"]) >= NUM_POINTS:
            sigs.append(np.array(rec["Z"][:NUM_POINTS], dtype=np.float32))
    
    sigs = np.array(sigs).reshape(-1, NUM_POINTS, 1)
    return feature_extractor.predict(sigs, batch_size=64)

print("🚀 Mengekstraksi fitur laten Indonesia...")
latents_indo = get_latents(JSON_INDO)

# Untuk referensi global, kita gunakan subset data STEAD yang Bapak miliki (UUSS/Global)
# Jika tidak ada, kita bisa gunakan noise murni hasil simulasi sebagai 'Anchor'
print("🚀 Mengekstraksi fitur laten Global (STEAD/Reference)...")
latents_global = get_latents("/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000/STEAD_5000_1C_20260719_062844.json")

# ==============================================================================
# 3. PROCRUSTES ALIGNMENT
# ==============================================================================
print("⚖️ Melakukan Feature Alignment dengan Procrustes...")

# Reduksi dimensi agar alignment stabil
pca = PCA(n_components=32)
ind_pca = pca.fit_transform(latents_indo)
glo_pca = pca.fit_transform(latents_global)

# Alignment: mtx1 adalah Global yang disesuaikan, mtx2 adalah Indonesia yang disesuaikan
mtx1, mtx2, disparity = procrustes(glo_pca, ind_pca)

print(f"✅ Alignment selesai! Disparity (Residual Error): {disparity:.6f}")

# ==============================================================================
# 4. APLIKASI HASIL ALIGNMENT
# ==============================================================================
# Sekarang, setiap data Indonesia baru (latents_indo_new) harus ditransformasi
# menggunakan matriks rotasi hasil Procrustes (mtx2)
def align_new_data(new_latents):
    new_pca = pca.transform(new_latents)
    # Proyeksi ke ruang yang sama (menggunakan matriks transformasi dari mtx2)
    # Dalam procrustes, alignment biasanya dicapai dengan rotasi/skala.
    return new_pca @ (mtx2.T @ mtx1) # Proyeksi sederhana

print("✅ Ruang laten kini selaras (Aligned). Model global siap digunakan.")

# Disini Bapak bisa lanjut ke inferensi KDE menggunakan 'aligned_target'
# yang nilainya sudah setara dengan ruang laten global (UUSS).

In [ ]:
# -*- coding: utf-8 -*-
"""
Evaluasi Inferensi Terkoreksi (Post-Procrustes Alignment)
"""

# 1. Hitung titik pusat (centroid) kelas Gempa dan Noise di ruang Global (UUSS)
# Ambil dari latents_global (karena tadi sudah di-align ke mtx1)
# Kita bagi berdasarkan label untuk mendapatkan representasi 'ideal'
from sklearn.metrics import accuracy_score

# Asumsi: Bapak punya cara membedakan label di latents_global (n_global)
# Misalnya dengan memetakan kembali ke label aslinya
def evaluate_aligned_inference(aligned_latents_indo, y_true):
    # Kita bandingkan jarak ke centroid Gempa vs Noise di ruang global yang sudah selaras
    # Jika titik laten indo lebih dekat ke centroid global gempa, maka = Gempa
    
    # 1. Hitung centroid global (dari mtx1)
    # mtx1 adalah latents_global yang sudah di-align
    # (Penting: Bapak harus memisahkan mtx1_gempa dan mtx1_noise)
    
    centroid_gempa = np.mean(mtx1[y_global == 1], axis=0) 
    centroid_noise = np.mean(mtx1[y_global == 0], axis=0)
    
    y_pred = []
    for latent in mtx2: # mtx2 adalah latents_indo yang sudah di-align
        dist_gempa = np.linalg.norm(latent - centroid_gempa)
        dist_noise = np.linalg.norm(latent - centroid_noise)
        y_pred.append(1 if dist_gempa < dist_noise else 0)
        
    acc = accuracy_score(y_true, y_pred)
    print(f"📊 Akurasi setelah Alignment: {acc*100:.2f}%")
    return y_pred

# Jalankan evaluasi
y_pred_aligned = evaluate_aligned_inference(mtx2, y_labels)

In [ ]:
import json
import numpy as np

def audit_json_quality(filepath):
    print(f"🔍 Audit dimulai: {filepath}")
    with open(filepath, 'r') as f:
        data = json.load(f)
        
    total = len(data)
    cacat_padding = 0
    cacat_nan = 0
    
    for key, record in data.items():
        z_sig = record.get('Z', [])
        # Cek NaN/Inf
        if any(np.isnan(z_sig)) or any(np.isinf(z_sig)):
            cacat_nan += 1
        # Cek Zero-Padding (10 titik terakhir nol semua)
        if len(z_sig) > 10 and all(abs(val) < 1e-6 for val in z_sig[-10:]):
            cacat_padding += 1
            
    print(f"Total: {total} | NaN: {cacat_nan} | Padding: {cacat_padding}")
    if cacat_padding > 0:
        print("⚠️ Rekomendasi: Bersihkan padding dengan skrip pembersih.")

# Jalankan audit
audit_json_quality('/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000/STEAD_5000_1C_20260719_062844.json')


In [ ]:
import json

def inspect_json_structure(filepath):
    with open(filepath, 'r') as f:
        # Membaca file sebagai iterator agar tidak membebani RAM
        data = json.load(f)
        
        # Ambil satu contoh record
        sample_key = next(iter(data))
        sample = data[sample_key]
        
        print(f"--- Struktur Data untuk Event: {sample_key} ---")
        for key in sample.keys():
            val = sample[key]
            if isinstance(val, list):
                print(f"Kolom: {key} | Tipe: List | Panjang: {len(val)}")
            else:
                print(f"Kolom: {key} | Tipe: {type(val).__name__} | Nilai: {val}")

inspect_json_structure('/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000/STEAD_5000_1C_CLEAN.json')

In [2]:
# -*- coding: utf-8 -*-
from Library import utils, dataset
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tqdm import tqdm
from datetime import datetime
import logging
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from sklearn.metrics import confusion_matrix
import seaborn as sns

# ==============================================================================
# 1. SETUP LOGGING & VISUALISASI (1C)
# ==============================================================================
def print_and_save_report(y_true, y_pred_before, y_pred_after, save_dir):
    from sklearn.metrics import classification_report
    
    # Hitung laporan
    report_before = classification_report(y_true, y_pred_before, output_dict=True)
    report_after = classification_report(y_true, y_pred_after, output_dict=True)
    
    # Simpan ke file
    report_data = {"Sebelum": report_before, "Sesudah": report_after}
    with open(os.path.join(save_dir, "classification_report_1c.json"), 'w') as f:
        json.dump(report_data, f, indent=4)
        
    print("\n" + "="*50)
    print("LAPORAN KLASIFIKASI: SEBELUM VS SESUDAH ADAPTASI")
    print("="*50)
    print(classification_report(y_true, y_pred_after, target_names=['Noise', 'Gempa']))
    print("="*50)

def plot_latent_distribution(le_global, no_local, save_dir):
    """Visualisasi apakah Noise Lokal sudah terpisah dengan Gempa Global"""
    x = np.linspace(-10, 5, 200)
    plt.figure(figsize=(10, 5))
    plt.plot(x, le_global.pdf(x), label='Gempa (Global/UUSS)', color='blue', linewidth=2)
    plt.plot(x, no_local.pdf(x), label='Noise (Lokal/STEAD)', color='red', linestyle='--', linewidth=2)
    plt.fill_between(x, no_local.pdf(x), color='red', alpha=0.1)
    plt.title('Distribusi Laten: Pergeseran Domain STEAD 1C', fontweight='bold')
    plt.xlabel('Nilai Laten (Feature Space)')
    plt.ylabel('Kepadatan Probabilitas (PDF)')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.savefig(os.path.join(save_dir, "latent_distribution_1c.jpg"), dpi=300)
    plt.close()

# ==============================================================================
# MAIN EXECUTION (1C)
# ==============================================================================
if __name__ == "__main__":
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

    # Konfigurasi STEAD 1C
    KEY_DATA_DIR = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_sample_5000'
    KEY_TEST_FILE = 'STEAD_5000_1C_CLEAN.json'  
    CALIBRATION_SIZE = 500
    DECISION_BIAS = 1.0

    MODEL_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Code & Figure demo/Pre-trained model/MCU-Quake 5-20"
    EMB_DIR = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Code & Figure demo/Typical embedding/Embedding_data train 3C, UUSS n11275 std15, 30120909"

    save_dir = os.path.join('/Volumes/Extreme SSD/stream_stead/output/stead_1c_da', datetime.now().strftime("%d%H%M%S"))
    os.makedirs(save_dir, exist_ok=True)

    logging.basicConfig(filename=os.path.join(save_dir, "log_1c.txt"), level=logging.INFO, format='%(asctime)s: %(message)s')
    logger = logging.getLogger()
    logger.addHandler(logging.StreamHandler())

    logger.info("Memuat model dan embedding 1C (Z-Component)...")
    model = keras.models.load_model(MODEL_PATH)
    emb_Z = dataset.load_embedding_data(EMB_DIR, "Embedding data, Z.json")
    
    # PDF Global Gempa (LE) 1C
    uuss_k_le = next(k for k in emb_Z.keys() if k.lower() in ['le','eq','earthquake'])
    kde_le_global = gaussian_kde(np.array(emb_Z[uuss_k_le]).flatten())

    # Load Data 1C
    test_data = dataset.load_json_data(os.path.join(KEY_DATA_DIR, KEY_TEST_FILE))
    keys = list(test_data.keys()); np.random.shuffle(keys)
    calib_keys, test_keys = keys[:CALIBRATION_SIZE], keys[CALIBRATION_SIZE:]

    # Kalibrasi Noise Lokal 1C
    calib_Z = []
    for k in tqdm(calib_keys, desc="Kalibrasi Noise Lokal 1C"):
        if "Z_noise" in test_data[k]:
            zn = np.array(test_data[k]["Z_noise"][-700:], dtype=np.float32)
            calib_Z.append(utils.latent_codes_1D(zn, model).flatten()[0])
            
    kde_no_local = gaussian_kde(calib_Z)

    # Inferensi 1C
    y_true, y_pred_before, y_pred_after = [], [], []
    for k in tqdm(test_keys, desc="Inferensi 1C"):
        rec = test_data[k]
        for is_gempa, comp_key in [(0, "Z_noise"), (1, "Z")]:
            if comp_key in rec:
                sig = np.array(rec[comp_key][:700], dtype=np.float32)
                z_feat = utils.latent_codes_1D(sig, model).flatten()[0]
                
                # Prediksi Sebelum (Menggunakan KDE Global bawaan)
                y_pred_before.append(1 if kde_le_global.pdf(z_feat) > 1.0 else 0) # Asumsi ambang 1.0
                # Prediksi Sesudah (Adaptasi Lokal)
                y_pred_after.append(1 if kde_le_global.pdf(z_feat) > (kde_no_local.pdf(z_feat) * DECISION_BIAS) else 0)
                y_true.append(is_gempa)

    plot_domain_adaptation_results_1c(y_true, y_pred_before, y_pred_after, save_dir)
    logger.info("Eksperimen 1C Selesai.")

    # 4. Evaluasi & Visualisasi Akhir
    logger.info("Menyusun laporan dan visualisasi akhir...")
    
    # Matriks Konfusi
    plot_domain_adaptation_results_1c(y_true, y_pred_before, y_pred_after, save_dir)
    
    # Distribusi Latent (Agar terlihat pergeseran domain-nya)
    plot_latent_distribution(kde_le_global, kde_no_local, save_dir)
    
    # Print & Save Laporan Angka
    print_and_save_report(y_true, y_pred_before, y_pred_after, save_dir)
    
    logger.info(f"Eksperimen 1C Selesai. Hasil ada di: {save_dir}")

Memuat model dan embedding 1C (Z-Component)...
Memuat model dan embedding 1C (Z-Component)...


No training configuration found in save file, so the model was *not* compiled. Compile it manually.
No training configuration found in save file, so the model was *not* compiled. Compile it manually.
Inferensi 1C: 100%|██████████| 3342/3342 [00:22<00:00, 148.23it/s]
Eksperimen 1C Selesai.
Eksperimen 1C Selesai.
Menyusun laporan dan visualisasi akhir...
Menyusun laporan dan visualisasi akhir...
Eksperimen 1C Selesai. Hasil ada di: /Volumes/Extreme SSD/stream_stead/output/stead_1c_da/25180917
Eksperimen 1C Selesai. Hasil ada di: /Volumes/Extreme SSD/stream_stead/output/stead_1c_da/25180917



LAPORAN KLASIFIKASI: SEBELUM VS SESUDAH ADAPTASI
              precision    recall  f1-score   support

       Noise       0.58      0.53      0.55      3342
       Gempa       0.57      0.61      0.59      3342

    accuracy                           0.57      6684
   macro avg       0.57      0.57      0.57      6684
weighted avg       0.57      0.57      0.57      6684

